# 4. Report

Tabel, grafik, evaluasi, dan contoh caption untuk laporan.


In [1]:
from pathlib import Path
import sys

def find_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start] + list(start.parents):
        if (path / 'src').exists() and (path / 'data').exists():
            return path
    raise RuntimeError('root repo tidak ditemukan')

ROOT = find_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('ROOT:', ROOT)


ROOT: /Users/rusmn/Kuliah/SEMESTER 6/Machine Learning/Tubes2/ML-Tubes-2_RecursiveLearnaholic


In [2]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from cnn.utility import image_loader
from common.metrics import score_set, bleu4_score
from rnn.caption_decoder import build_decoder

PROC_DIR = ROOT / 'data/processed/flickr8k'
RAW_IMAGE_DIR = ROOT / 'data/raw/flickr8k/Images'
MODEL_DIR = ROOT / 'models/rnn'
TABLE_DIR = ROOT / 'reports/tables/rnn'
FIG_DIR = ROOT / 'reports/figures/rnn'
for folder in [TABLE_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

KERAS_BATCH = 128
SCRATCH_BATCH = 32
RAW_DEMO_COUNT = 5

print('report: hitung ulang')


report: hitung ulang


In [3]:
def artifact_path(value):
    path = Path(str(value))
    return path if path.is_absolute() else ROOT / path

def repo_relative(value):
    path = artifact_path(value)
    try:
        return path.resolve().relative_to(ROOT.resolve()).as_posix()
    except Exception:
        return str(value)

def load_json(path, default=None):
    path = Path(path)
    if path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    return default

def save_json(value, path):
    Path(path).write_text(json.dumps(value, indent=2), encoding='utf-8')

def load_records():
    records = load_json(TABLE_DIR / 'train_records.json', [])
    for record in records:
        record['model_path'] = repo_relative(record['model_path'])
        record['weight_path'] = repo_relative(record['weight_path'])
        record['history_path'] = repo_relative(record['history_path'])
    return records

word_to_index = json.loads((PROC_DIR / 'vocab.json').read_text(encoding='utf-8'))
index_to_word = {int(value): key for key, value in word_to_index.items()}
train_records = load_records()
print('record:', len(train_records))


record: 12


In [4]:
test_features_all = np.load(PROC_DIR / 'test_features.npy').astype('float32')
test_captions_all = np.load(PROC_DIR / 'test_captions.npy').astype('int32')
test_ids_all = [line.strip() for line in (PROC_DIR / 'test_image_ids.txt').read_text(encoding='utf-8').splitlines() if line.strip()]

def seq_words(seq):
    words = []
    for token in np.asarray(seq).astype(int).tolist():
        word = index_to_word.get(token, '<unk>')
        if word in {'<start>', '<pad>'}:
            continue
        if word == '<end>':
            break
        words.append(word)
    return words

# Satu baris evaluasi per image; semua caption referensi image tersebut digabung.
group = {}
for feature, seq, image_id in zip(test_features_all, test_captions_all, test_ids_all):
    if image_id not in group:
        group[image_id] = {'feature': feature, 'refs': []}
    group[image_id]['refs'].append(seq_words(seq))
image_ids = sorted(group)
eval_features = np.asarray([group[image_id]['feature'] for image_id in image_ids], dtype='float32')
eval_refs = [group[image_id]['refs'] for image_id in image_ids]
print('test:', len(image_ids))


test: 1012


In [5]:
def decode_keras(model, features, max_length, batch_size=128):
    pad = word_to_index['<pad>']; start = word_to_index['<start>']; end = word_to_index['<end>']
    outputs = []
    for left in range(0, len(features), batch_size):
        batch = features[left:left+batch_size]
        n = len(batch)
        seq = np.full((n, max_length), pad, dtype='int32')
        gen = np.full((n, max_length), pad, dtype='int32')
        seq[:, 0] = start
        done = np.zeros(n, dtype=bool)
        for step in range(max_length):
            probs = model([batch, seq], training=False).numpy()
            next_ids = np.argmax(probs[:, step, :], axis=-1).astype('int32')
            next_ids[done] = pad
            gen[:, step] = next_ids
            done |= next_ids == end
            if step + 1 < max_length:
                seq[:, step + 1] = next_ids
            if done.all():
                break
        outputs.extend(seq_words(row) for row in gen)
    return outputs

def decode_scratch(decoder, features, max_length, batch_size=32):
    pad = word_to_index['<pad>']; start = word_to_index['<start>']; end = word_to_index['<end>']
    outputs = []
    for left in range(0, len(features), batch_size):
        batch = features[left:left+batch_size]
        n = len(batch)
        seq = np.full((n, max_length), pad, dtype='int32')
        gen = np.full((n, max_length), pad, dtype='int32')
        seq[:, 0] = start
        done = np.zeros(n, dtype=bool)
        for step in range(max_length):
            probs = decoder.predict(batch, seq)
            next_ids = np.argmax(probs[:, step, :], axis=-1).astype('int32')
            next_ids[done] = pad
            gen[:, step] = next_ids
            done |= next_ids == end
            if step + 1 < max_length:
                seq[:, step + 1] = next_ids
            if done.all():
                break
        outputs.extend(seq_words(row) for row in gen)
    return outputs

def score_predictions(predictions, elapsed):
    score = score_set(eval_refs, predictions)
    score['runtime_seconds'] = float(elapsed)
    score['samples'] = len(predictions)
    return score


In [6]:
score_path = TABLE_DIR / 'rnn_lstm_results.json'
scores = []
for record in train_records:
    cfg = record['config']
    model_path = repo_relative(record['model_path'])
    print('evaluasi:', Path(model_path).name)
    model = tf.keras.models.load_model(artifact_path(model_path), safe_mode=False)
    t0 = time.time()
    preds = decode_keras(model, eval_features, int(cfg['caption_length']), KERAS_BATCH)
    elapsed = time.time() - t0
    row = {
        **score_predictions(preds, elapsed),
        'implementation': 'keras',
        'recur_type': cfg['recur_type'],
        'recur_layers': int(cfg['recur_layers']),
        'hidden_size': int(cfg['hidden_size']),
        'model_path': model_path,
        'weight_path': repo_relative(record['weight_path']),
    }
    scores.append(row)
    save_json(scores, score_path)
    pd.DataFrame(scores).to_csv(TABLE_DIR / 'rnn_lstm_results.csv', index=False)

keras_df = pd.DataFrame(scores)
keras_df.to_csv(TABLE_DIR / 'rnn_lstm_results.csv', index=False)
keras_df.sort_values('bleu4', ascending=False).head()


evaluasi: rnn_layers1_hidden128_len33.keras


KeyboardInterrupt: 

In [ ]:
best_by_type = {}
for kind in ['rnn', 'lstm']:
    part = keras_df[keras_df['recur_type'] == kind]
    if len(part):
        best_by_type[kind] = part.sort_values(['bleu4', 'meteor'], ascending=False).iloc[0].to_dict()
print({k: Path(v['model_path']).name for k, v in best_by_type.items()})

scratch_path = TABLE_DIR / 'scratch_best_results.json'
scratch_scores = []
for kind, best in best_by_type.items():
    cfg = {
        'vocab_size': len(word_to_index),
        'feature_dim': int(eval_features.shape[1]),
        'embed_dim': 256,
        'hidden_size': int(best['hidden_size']),
        'recur_layers': int(best['recur_layers']),
        'recur_type': kind,
    }
    print('scratch:', kind)
    decoder = build_decoder(cfg, artifact_path(best['weight_path']))
    t0 = time.time()
    preds = decode_scratch(decoder, eval_features, int(test_captions_all.shape[1] - 1), SCRATCH_BATCH)
    elapsed = time.time() - t0
    row = {
        **score_predictions(preds, elapsed),
        'implementation': 'scratch',
        'recur_type': kind,
        'recur_layers': int(best['recur_layers']),
        'hidden_size': int(best['hidden_size']),
        'model_path': repo_relative(best['model_path']),
        'weight_path': repo_relative(best['weight_path']),
    }
    scratch_scores.append(row)
    save_json(scratch_scores, scratch_path)
    pd.DataFrame(scratch_scores).to_csv(TABLE_DIR / 'scratch_best_results.csv', index=False)

scratch_df = pd.DataFrame(scratch_scores)
scratch_df.to_csv(TABLE_DIR / 'scratch_best_results.csv', index=False)
comparison = pd.concat([pd.DataFrame(best_by_type.values()), scratch_df], ignore_index=True)
comparison.to_csv(TABLE_DIR / 'keras_vs_scratch.csv', index=False)
comparison


In [ ]:
keras_df.groupby(['recur_type', 'recur_layers'])[['bleu4', 'meteor', 'runtime_seconds']].mean().reset_index().to_csv(TABLE_DIR / 'caption_by_recurrent_layers.csv', index=False)
keras_df.groupby(['recur_type', 'hidden_size'])[['bleu4', 'meteor', 'runtime_seconds']].mean().reset_index().to_csv(TABLE_DIR / 'caption_by_hidden_size.csv', index=False)
keras_df.groupby('recur_type')[['bleu4', 'meteor', 'runtime_seconds']].mean().reset_index().to_csv(TABLE_DIR / 'rnn_vs_lstm.csv', index=False)
keras_df.to_csv(TABLE_DIR / 'caption_all_keras_variations.csv', index=False)

plot_scores = keras_df.copy().sort_values(['recur_type', 'recur_layers', 'hidden_size'])
plot_scores['label'] = plot_scores.apply(lambda row: f"{row['recur_type'].upper()} L{int(row['recur_layers'])} H{int(row['hidden_size'])}", axis=1)

plt.figure(figsize=(12, 4.5))
plt.bar(plot_scores['label'], plot_scores['bleu4'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('BLEU-4')
plt.title('BLEU-4 Semua Decoder')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'decoder_bleu4.png', dpi=150)
plt.savefig(FIG_DIR / 'rnn_lstm_bleu4_variations.png', dpi=150)
plt.show()

plt.figure(figsize=(12, 4.5))
plt.bar(plot_scores['label'], plot_scores['meteor'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('METEOR')
plt.title('METEOR Semua Decoder')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'decoder_meteor.png', dpi=150)
plt.savefig(FIG_DIR / 'rnn_lstm_meteor_variations.png', dpi=150)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for metric, row_axes in [('bleu4', axes[0]), ('meteor', axes[1])]:
    keras_df.groupby(['recur_type', 'recur_layers'])[metric].mean().unstack(0).plot(marker='o', ax=row_axes[0])
    row_axes[0].set_title(f'{metric.upper()} vs Layer')
    row_axes[0].set_ylabel(metric.upper())
    row_axes[0].grid(alpha=0.3)
    keras_df.groupby(['recur_type', 'hidden_size'])[metric].mean().unstack(0).plot(marker='o', ax=row_axes[1])
    row_axes[1].set_title(f'{metric.upper()} vs Hidden')
    row_axes[1].set_ylabel(metric.upper())
    row_axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'rnn_layer_hidden_effects.png', dpi=150)
plt.show()

plt.figure(figsize=(10, 5))
for hist_path in sorted(TABLE_DIR.glob('*_history.json')):
    history = json.loads(hist_path.read_text(encoding='utf-8'))
    label = hist_path.stem.replace('_history', '')
    values = history.get('loss', [])
    if values:
        plt.plot(values, alpha=0.45, label=label)
plt.title('Training Loss Semua Decoder')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'training_loss.png', dpi=150)
plt.savefig(FIG_DIR / 'training_loss_curves.png', dpi=150)
plt.show()

plt.figure(figsize=(10, 5))
for hist_path in sorted(TABLE_DIR.glob('*_history.json')):
    history = json.loads(hist_path.read_text(encoding='utf-8'))
    label = hist_path.stem.replace('_history', '')
    values = history.get('val_loss', [])
    if values:
        plt.plot(values, alpha=0.45, label=label)
plt.title('Validation Loss Semua Decoder')
plt.xlabel('Epoch')
plt.ylabel('Val loss')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'validation_loss.png', dpi=150)
plt.savefig(FIG_DIR / 'validation_loss_curves.png', dpi=150)
plt.show()


In [ ]:
length_path = TABLE_DIR / 'length_scores.json'
length_scores = []
all_candidates = list(best_by_type.values()) + scratch_scores
best_overall = sorted(all_candidates, key=lambda row: (row.get('bleu4', 0), row.get('meteor', 0)), reverse=True)[0]
print('best:', best_overall['implementation'], best_overall['recur_type'])
for length in [10, 20, int(test_captions_all.shape[1] - 1)]:
    print('length:', length)
    t0 = time.time()
    if best_overall['implementation'] == 'scratch':
        cfg = {
            'vocab_size': len(word_to_index), 'feature_dim': int(eval_features.shape[1]), 'embed_dim': 256,
            'hidden_size': int(best_overall['hidden_size']), 'recur_layers': int(best_overall['recur_layers']), 'recur_type': best_overall['recur_type']
        }
        decoder = build_decoder(cfg, artifact_path(best_overall['weight_path']))
        preds = decode_scratch(decoder, eval_features, length, SCRATCH_BATCH)
    else:
        model = tf.keras.models.load_model(artifact_path(best_overall['model_path']), safe_mode=False)
        preds = decode_keras(model, eval_features, length, KERAS_BATCH)
    elapsed = time.time() - t0
    row = {**score_predictions(preds, elapsed), 'max_length': int(length), 'implementation': best_overall['implementation'], 'recur_type': best_overall['recur_type']}
    length_scores.append(row)
    save_json(length_scores, length_path)

length_df = pd.DataFrame(length_scores)
length_df.to_csv(TABLE_DIR / 'length_scores.csv', index=False)
length_df.plot(x='max_length', y='bleu4', marker='o', legend=False, title='Caption Length Study')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'length_bleu4.png', dpi=150)
plt.show()
length_df


In [ ]:
qual_path = TABLE_DIR / 'qualitative_examples.json'
sample_count = len(eval_features)
sample_features = eval_features[:sample_count]
sample_refs = eval_refs[:sample_count]
sample_ids = image_ids[:sample_count]
preds_by_kind = {}
for kind, best in best_by_type.items():
    print('qual:', kind)
    model = tf.keras.models.load_model(artifact_path(best['model_path']), safe_mode=False)
    preds_by_kind[kind] = decode_keras(model, sample_features, int(test_captions_all.shape[1] - 1), KERAS_BATCH)
avg_bleu = []
for idx in range(sample_count):
    vals = [bleu4_score(sample_refs[idx], preds[idx]) for preds in preds_by_kind.values()]
    avg_bleu.append(float(np.mean(vals)))
order = np.argsort(avg_bleu)
selected = []
for pool in np.array_split(order, 3):
    selected.extend(pool[:4].tolist())
selected = selected[:10]
qualitative = []
for idx in selected:
    qualitative.append({
        'image_id': sample_ids[idx],
        'avg_bleu4': avg_bleu[idx],
        'ground_truth': [' '.join(ref) for ref in sample_refs[idx]],
        'rnn_prediction': ' '.join(preds_by_kind.get('rnn', [[]])[idx]) if 'rnn' in preds_by_kind else '',
        'lstm_prediction': ' '.join(preds_by_kind.get('lstm', [[]])[idx]) if 'lstm' in preds_by_kind else '',
    })
save_json(qualitative, qual_path)
pd.DataFrame(qualitative).to_csv(TABLE_DIR / 'qualitative_examples.csv', index=False)
pd.DataFrame(qualitative)


In [ ]:
raw_demo_path = TABLE_DIR / 'raw_caption_examples.json'
if not RAW_IMAGE_DIR.exists():
    print('raw image belum ada')
    raw_rows = []
else:
    cached_weights = Path.home() / '.keras/models/inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5'
    weights_arg = str(cached_weights) if cached_weights.exists() else 'imagenet'
    encoder = tf.keras.applications.InceptionV3(include_top=False, weights=weights_arg, pooling='avg')
    encoder.trainable = False
    demo_ids = [row.get('image_id') for row in qualitative if row.get('image_id')][:RAW_DEMO_COUNT]
    if not demo_ids:
        demo_ids = image_ids[:RAW_DEMO_COUNT]
    decoders = {}
    for kind, best in best_by_type.items():
        cfg = {
            'vocab_size': len(word_to_index),
            'feature_dim': int(eval_features.shape[1]),
            'embed_dim': 256,
            'hidden_size': int(best['hidden_size']),
            'recur_layers': int(best['recur_layers']),
            'recur_type': kind,
        }
        decoders[kind] = build_decoder(cfg, artifact_path(best['weight_path']))

    raw_rows = []
    for image_id in demo_ids:
        print('raw:', image_id)
        image_path = RAW_IMAGE_DIR / image_id
        if not image_path.exists():
            continue
        image = image_loader(image_path, target_size=(299, 299))
        image = tf.keras.applications.inception_v3.preprocess_input(image[None, ...] * 255.0)
        feature = encoder.predict(image, verbose=0).astype('float32')
        row = {'image_id': image_id}
        for kind, decoder in decoders.items():
            pred = decode_scratch(decoder, feature, int(test_captions_all.shape[1] - 1), batch_size=1)[0]
            row[f'{kind}_scratch_caption'] = ' '.join(pred)
        raw_rows.append(row)
    save_json(raw_rows, raw_demo_path)

pd.DataFrame(raw_rows).to_csv(TABLE_DIR / 'raw_caption_examples.csv', index=False)
pd.DataFrame(raw_rows)
